# 02-1. 시계열 파운데이션 모델(Chronos-2) 예측

**Chronos-2**는 아마존이 공개한 시계열 예측 모델입니다. 수많은 종류의 시계열로 미리 학습되어 있어서, **우리 데이터로 따로 학습하지 않고도** 과거 기록만 넣으면 바로 예측합니다(사전학습 모델, 라이선스 Apache-2.0).
또한 '내일 가동하는가'처럼 **미래에 미리 아는 정보(공변량)**를 함께 넣을 수 있습니다.

이 노트북은 02-2 노트북과 **같은 5주 시험 기간**에 대해 Chronos-2의 예측값을 만들어 저장합니다. 비교와 해석은 02-2 노트북에서 합니다.

> **실행 환경**: 다른 노트북과 같은 환경(`requirements.txt`, Python 3.11)에서 실행합니다. 이번 실행은 CPU만으로 약 24분 걸렸습니다. `run_all.py`는 기본으로 이 노트북을 건너뛰고 저장된 예측을 쓰며, `--with-chronos`를 주면 다시 계산합니다.
> 처음 실행할 때 인터넷에서 모델(약 120M 파라미터)을 내려받습니다.

## 0. 준비

In [1]:
CHRONOS_REVISION = "29ec3766d36d6f73f0696f85560a422f50e8498c"   # 결과 재현을 위해 모델 가중치 버전 고정 (Hugging Face 커밋)
import time, warnings
warnings.filterwarnings("ignore")   # 패키지를 불러올 때 나오는 경고도 숨김 (설치 경로가 출력에 남지 않게)
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from chronos import Chronos2Pipeline

torch.manual_seed(42)
PRED_DIR = Path("../outputs/predictions"); PRED_DIR.mkdir(parents=True, exist_ok=True)

slots = pd.read_csv("../data/processed/slots_15min.csv", encoding="utf-8-sig", parse_dates=["ts", "date"], index_col="ts")
slots["칸번호"] = (slots["시각"] * 4).astype(int)
slots["요일"] = slots.index.dayofweek
slots["하루종류코드"] = slots["하루종류"].map({"평일가동": 0, "토요일": 1, "휴일": 2})
slots["전날가동"] = slots["전날가동"].astype(int)

TEST_STARTS = pd.to_datetime(["2021-08-09", "2021-08-16", "2021-08-23", "2021-08-30", "2021-09-06"])
TEST_ENDS = list(TEST_STARTS[1:]) + [pd.Timestamp("2021-09-15")]
CURVE_KEYS = ["하루종류코드", "전날가동", "칸번호"]
PLAN = ["하루종류코드", "전날가동", "칸번호", "요일"]       # 미래에 미리 아는 정보 (달력 + 가동 계획)
QUANTILES = [0.1, 0.5, 0.9]

pipe = Chronos2Pipeline.from_pretrained("amazon/chronos-2", revision=CHRONOS_REVISION, device_map="cpu")
print("모델 준비 완료 | torch", torch.__version__)

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Loading weights:  29%|██▉       | 50/170 [00:00<00:00, 481.52it/s]

Loading weights: 100%|██████████| 170/170 [00:00<00:00, 976.76it/s]

모델 준비 완료 | torch 2.14.1+cpu


## 1. 하루 앞 예측
매일 자정 직전에, 7/1부터 전날까지의 전력 기록을 넣고 다음 날 96칸을 예측합니다. 세 가지 방식으로 넣어 봅니다.

| 이름 | 넣는 정보 |
|---|---|
| Chronos-2 (기록만) | 과거 전력 기록만 |
| Chronos-2 (+가동 계획) | 과거 기록 + 시각·요일·하루 종류·전날 가동 여부 |
| Chronos-2 (+기준 곡선) | 위 정보 + 02-2 노트북의 기준 곡선 값 |

In [2]:
def make_task(item, ctx, fut, cov):
    df = pd.DataFrame({"item_id": item, "timestamp": ctx.index, "target": ctx["전력"].values})
    fdf = pd.DataFrame({"item_id": item, "timestamp": fut.index})
    for c in cov:
        df[c] = ctx[c].values.astype(float)
        fdf[c] = fut[c].values.astype(float)
    return df, fdf

VARIANTS = {"Chronos-2 (기록만)": [], "Chronos-2 (+가동 계획)": PLAN, "Chronos-2 (+기준 곡선)": PLAN + ["기준곡선"]}
t0 = time.time()
day_out = []
for name, cov in VARIANTS.items():
    dfs, fdfs, futs = [], [], []
    for st, en in zip(TEST_STARTS, TEST_ENDS):
        curve = slots[(slots.index < st) & slots["정답사용"]].groupby(CURVE_KEYS)["전력"].mean().rename("기준곡선")
        ss = slots.join(curve, on=CURVE_KEYS)
        for day in pd.date_range(st, en - pd.Timedelta(days=1)):
            ctx = ss[ss.index < day]
            fut = ss[(ss.index >= day) & (ss.index < day + pd.Timedelta(days=1))]
            df, fdf = make_task(f"{day:%m%d}", ctx, fut, cov)
            dfs.append(df); fdfs.append(fdf); futs.append(fut)
    pred = pipe.predict_df(pd.concat(dfs), future_df=pd.concat(fdfs) if cov else None,
                           prediction_length=96, quantile_levels=QUANTILES, batch_size=64)
    pred = pred.rename(columns={"timestamp": "ts", "predictions": "예측", "0.1": "하위10%", "0.5": "중앙값", "0.9": "상위10%"})
    pred["모델"] = name
    day_out.append(pred[["ts", "모델", "예측", "하위10%", "중앙값", "상위10%"]])
    print(f"{name}: {len(pred):,}칸 예측, 누적 {time.time() - t0:.0f}초")
chronos_day = pd.concat(day_out)
chronos_day.to_csv(PRED_DIR / "chronos_day_ahead.csv", encoding="utf-8-sig", index=False)

Chronos-2 (기록만): 3,552칸 예측, 누적 15초


Chronos-2 (+가동 계획): 3,552칸 예측, 누적 76초


Chronos-2 (+기준 곡선): 3,552칸 예측, 누적 147초


## 2. 1시간 앞 예측
매 정시에, 직전 3주(2,016칸)의 전력 기록과 가동 계획을 넣고 다음 4칸을 예측합니다.
(모델에 넣는 과거 기록을 3주로 줄인 것은 CPU 계산 시간 때문입니다. 1주 패턴을 세 번 볼 수 있는 길이입니다.)

In [3]:
CTX = 2016
t0 = time.time()
hour_out = []
for st, en in zip(TEST_STARTS, TEST_ENDS):
    dfs, fdfs = [], []
    for t in pd.date_range(st, en - pd.Timedelta(hours=1), freq="h"):
        ctx = slots[slots.index < t].iloc[-CTX:]
        fut = slots.loc[t:t + pd.Timedelta(minutes=45)]
        df, fdf = make_task(f"{t:%m%d%H}", ctx, fut, PLAN)
        dfs.append(df); fdfs.append(fdf)
    pred = pipe.predict_df(pd.concat(dfs), future_df=pd.concat(fdfs), prediction_length=4,
                           quantile_levels=QUANTILES, batch_size=128)
    pred["발행시각"] = pd.to_datetime("2021" + pred["item_id"], format="%Y%m%d%H")
    hour_out.append(pred)
    print(f"{st:%m/%d} 주 완료, 누적 {time.time() - t0:.0f}초")
chronos_hour = pd.concat(hour_out).rename(columns={"timestamp": "대상시각", "predictions": "예측", "0.1": "하위10%", "0.5": "중앙값", "0.9": "상위10%"})
chronos_hour = chronos_hour[["발행시각", "대상시각", "예측", "하위10%", "중앙값", "상위10%"]]
chronos_hour.to_csv(PRED_DIR / "chronos_hour_ahead.csv", encoding="utf-8-sig", index=False)
print("저장 완료:", sorted(p.name for p in PRED_DIR.glob("chronos_*.csv")))

08/09 주 완료, 누적 91초


08/16 주 완료, 누적 182초


08/23 주 완료, 누적 273초


08/30 주 완료, 누적 363초


09/06 주 완료, 누적 479초
저장 완료: ['chronos_day_ahead.csv', 'chronos_hour_ahead.csv', 'chronos_hour_q21.csv']


## 3. 저장 결과
- `outputs/predictions/chronos_day_ahead.csv`: 하루 앞 예측 (3가지 방식)
- `outputs/predictions/chronos_hour_ahead.csv`: 1시간 앞 예측

예측값과 함께 **하위 10%·상위 10% 값**도 저장했습니다. 실제 전력이 이 범위 안에 들어올 것이라는 모델의 판단으로, 02-3 노트북의 피크 경보에서 활용할 수 있습니다.

## 4. 1시간 앞 예측구간용 — 21개 분위수, 7/12부터
02-3 노트북 5-2절의 **예측구간**에 쓰는 자료입니다. 2절과 같은 방식(직전 3주 기록 + 가동 계획, 매 정시 다음 4칸)으로 예측하되, 두 가지를 바꿉니다.
- **분위수 21개**(1%, 5%, 10%, …, 95%, 99%)를 저장합니다. 80% 구간(10~90%)과 90% 구간(5~95%)을 모두 만들 수 있습니다.
- **7/12부터** 예측합니다. 시험 기간(8/9~) 이전 4주의 예측과 실제의 차이를 모아야, 1주차부터 구간을 보정할 수 있기 때문입니다.

같은 모델·같은 입력이므로 시험 기간의 평균 예측은 2절의 결과와 같아야 합니다(아래에서 확인). CPU로 약 20분 걸립니다.

In [4]:
QS21 = [0.01] + [round(0.05 * i, 2) for i in range(1, 20)] + [0.99]
Q_COLS = {str(q): f"q{round(q * 100):02d}" for q in QS21}            # 0.05 → q05
t0 = time.time()
q_out = []
for st in pd.date_range("2021-07-12", "2021-09-13", freq="7D"):
    en = min(st + pd.Timedelta(days=7), pd.Timestamp("2021-09-15"))
    dfs, fdfs = [], []
    for t in pd.date_range(st, en - pd.Timedelta(hours=1), freq="h"):
        ctx = slots[slots.index < t].iloc[-CTX:]
        fut = slots.loc[t:t + pd.Timedelta(minutes=45)]
        df, fdf = make_task(f"{t:%m%d%H}", ctx, fut, PLAN)
        dfs.append(df); fdfs.append(fdf)
    pred = pipe.predict_df(pd.concat(dfs), future_df=pd.concat(fdfs), prediction_length=4,
                           quantile_levels=QS21, batch_size=128)
    pred["발행시각"] = pd.to_datetime("2021" + pred["item_id"], format="%Y%m%d%H")
    q_out.append(pred)
    print(f"{st:%m/%d} 주 완료, 누적 {time.time() - t0:.0f}초")
chronos_q21 = pd.concat(q_out).rename(columns={"timestamp": "대상시각", "predictions": "예측", **Q_COLS})
chronos_q21 = chronos_q21[["발행시각", "대상시각", "예측"] + list(Q_COLS.values())]
chronos_q21.to_csv(PRED_DIR / "chronos_hour_q21.csv", encoding="utf-8-sig", index=False)

# 확인: 시험 기간의 평균 예측과 10%·90% 분위수가 2절 결과와 같은가
chk = chronos_hour.merge(chronos_q21, on=["발행시각", "대상시각"], suffixes=("", "_q21"))
gap = max((chk["예측"] - chk["예측_q21"]).abs().max(), (chk["하위10%"] - chk["q10"]).abs().max(), (chk["상위10%"] - chk["q90"]).abs().max())
assert len(chk) == len(chronos_hour) and gap < 1e-3
print(f"저장: chronos_hour_q21.csv ({len(chronos_q21):,}행, {chronos_q21['발행시각'].min():%m/%d}~{chronos_q21['발행시각'].max():%m/%d}) | 2절 결과와 최대 차이 {gap:.0e} (같음)")

07/12 주 완료, 누적 64초


07/19 주 완료, 누적 154초


07/26 주 완료, 누적 245초


08/02 주 완료, 누적 335초


08/09 주 완료, 누적 424초


08/16 주 완료, 누적 512초


08/23 주 완료, 누적 602초


08/30 주 완료, 누적 691초


09/06 주 완료, 누적 781초


09/13 주 완료, 누적 807초
저장: chronos_hour_q21.csv (6,240행, 07/12~09/14) | 2절 결과와 최대 차이 5e-05 (같음)
